In [ ]:
from platform import python_version
print(python_version())

# Detecting Mouse faces pain/not pain using PyTorch

Code copied from:
Image classification of Chest X Rays in one of three classes: Normal, Viral Pneumonia, COVID-19

Notebook created for the guided project [Detecting COVID-19 with Chest X Ray using PyTorch](https://www.coursera.org/projects/covid-19-detection-x-ray) on Coursera

Dataset from [COVID-19 Radiography Dataset](https://www.kaggle.com/tawsifurrahman/covid19-radiography-database) on Kaggle

# Importing Libraries

In [ ]:
# !pip3 install tqdm

In [ ]:
%matplotlib inline

import os, sys, copy
import shutil
import random
import torch
import torchvision
import numpy as np

from PIL import Image
from matplotlib import pyplot as plt

sys.path.insert(1, '../src/')

from Basic import *

pd.set_option("display.precision", 3)
from IPython.display import display, HTML
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

torch.manual_seed(0)
print('Using PyTorch version', torch.__version__)

### Preparing Training and Test Sets

#### HEIC to JPG - https://freetoolonline.com/heic-to-jpg.html

In [ ]:
root0 = '/media/flalix/d2f268d1-512d-499f-b3b3-6dad7d3fdd25/colaboracoes/deOcesano'
root_data = os.path.join(root0, "classifiers")
type_image = 'tif'; num_test_imgs = 15

class_names = ['10pct', '1pct', '1pct+il1b']
source_dirs = ['24h - 20x - 10% SFB', '24h - 20x - 1% SFB', '24h - 20x - 1% SFB and IL-1B']

root_ten_pct = os.path.join(root_data, source_dirs[0])
root_one_pct = os.path.join(root_data, source_dirs[1])
root_inflamm = os.path.join(root_data, source_dirs[2])

root_backup = os.path.join(root_data, 'backup')
root_backup_ten_pct = os.path.join(root_backup, source_dirs[0])
root_backup_one_pct = os.path.join(root_backup, source_dirs[1])
root_backup_inflamm = os.path.join(root_backup, source_dirs[2])

root_train         = os.path.join(root_data, 'train')
root_train_ten_pct = os.path.join(root_train, source_dirs[0])
root_train_one_pct = os.path.join(root_train, source_dirs[1])
root_train_inflamm = os.path.join(root_train, source_dirs[2])

root_test         = os.path.join(root_data, 'test')
root_test_ten_pct = os.path.join(root_test, source_dirs[0])
root_test_one_pct = os.path.join(root_test, source_dirs[1])
root_test_inflamm = os.path.join(root_test, source_dirs[2])

class_names == source_dirs, type_image, num_test_imgs, root_ten_pct, root_one_pct, root_inflamm, root_test

In [ ]:
dirs = [root_ten_pct, root_one_pct, root_inflamm]
for dir in dirs:
    print(os.path.exists(dir), dir)

In [ ]:
dirs = [root_backup, root_backup_ten_pct, root_backup_one_pct, root_backup_inflamm]

for dir in dirs:
    create_dir(dir)
    print(os.path.exists(dir), dir)

In [ ]:
dirs = [root_train, root_train_ten_pct, root_train_one_pct, root_train_inflamm]

for dir in dirs:
    create_dir(dir)
    print(os.path.exists(dir), dir)

In [ ]:
dirs = [root_test, root_test_ten_pct, root_test_one_pct, root_test_inflamm]

for dir in dirs:
    create_dir(dir)
    print(os.path.exists(dir), dir)

In [ ]:
def clean_data(rooti, verbose:bool=False):
    try:
        shutil.rmtree(rooti)
    except:
        pass

    try:
        os.mkdir(rooti)
    except:
        print("Impossible to creater", rooti)

dirs = [root_train_ten_pct, root_train_one_pct, root_train_inflamm]

for dir in dirs:
    clean_data(dir)
    fnames = os.listdir(dir)
    print(os.path.exists(dir), len(fnames), dir)

dirs = [root_test_ten_pct, root_test_one_pct, root_test_inflamm]

for dir in dirs:
    clean_data(dir)
    fnames = os.listdir(dir)
    print(os.path.exists(dir), len(fnames), dir)


### How many samples

In [ ]:
n_tenperc = len(os.listdir(root_ten_pct))
n_oneperc = len(os.listdir(root_one_pct))
n_inflamm = len(os.listdir(root_inflamm))

n_tenperc, n_oneperc, n_inflamm

### Sample data

In [ ]:
def copy_backup_to_data(root_backup_ctrl:str, root_train_ctrl:str, root_test_ctrl:str,
                        root_backup_case:str, root_train_case:str, root_test_case:str, 
                        perc_samples:float=.60, type_image:str='tif'):
    
   
    #----------- control --------------------
    backup_ctrl_fnames = [x for x in os.listdir(root_backup_ctrl) if x.lower().endswith(type_image)]

    n = len(backup_ctrl_fnames)  
    sele = list(np.random.choice(n, size=int(n*perc_samples), replace=False))
    sele.sort()
        
    not_sele = [x for x in np.arange(0, n) if x not in sele]
    not_sele.sort()

    fnames = [x for x in os.listdir(root_backup_ctrl) if x.lower().endswith(type_image)]
    fname_train_list = np.array(fnames)[sele]
    fname_test_list  = np.array(fnames)[not_sele]

    print(n, len(sele), len(not_sele) )

    for fname in fname_train_list:
        source_path = os.path.join(root_backup_ctrl, fname)
        target_path = os.path.join(root_train_ctrl, fname)
        shutil.copy(source_path, target_path) 

    for fname in fname_test_list:
        source_path = os.path.join(root_backup_ctrl, fname)
        target_path = os.path.join(root_test_ctrl, fname)
        shutil.copy(source_path, target_path) 

    #--------- case - inflammation -----------------------------
    backup_case_fnames = [x for x in os.listdir(root_backup_case) if x.lower().endswith(type_image)]

    n = len(backup_case_fnames)  
    sele = list(np.random.choice(n, size=int(n*perc_samples), replace=False))
    sele.sort()
        
    not_sele = [x for x in np.arange(0, n) if x not in sele]
    not_sele.sort()

    print(n, len(sele), len(not_sele) )
    
    fnames = [x for x in os.listdir(root_backup_case) if x.lower().endswith(type_image)]
    fname_train_list = np.array(fnames)[sele]
    fname_test_list  = np.array(fnames)[not_sele]
    
    for fname in fname_train_list:
        source_path = os.path.join(root_backup_case, fname)
        target_path = os.path.join(root_train_case, fname)
        shutil.copy(source_path, target_path) 

    for fname in fname_test_list:
        source_path = os.path.join(root_backup_case, fname)
        target_path = os.path.join(root_test_case, fname)
        shutil.copy(source_path, target_path) 

root_backup_ctrl=root_backup_one_pct
root_backup_case=root_backup_inflamm

root_train_ctrl=root_train_one_pct
root_test_ctrl=root_test_one_pct

root_train_case=root_train_inflamm
root_test_case=root_test_inflamm

copy_backup_to_data(root_backup_ctrl, root_train_ctrl, root_test_ctrl, 
                    root_backup_case, root_train_case, root_test_case, 
                    perc_samples=.60, type_image='tif')


In [ ]:
n=150
perc_samples=0.6

sele = list(np.random.choice(n, size=int(n*perc_samples), replace=False))
len(sele), len(np.unique(sele))

# Creating Custom Dataset

### Original size

In [ ]:
fnames = [x for x in os.listdir(root_train_ctrl) ]

img = Image.open(os.path.join(root_train_ctrl, fnames[0]))
width, height = img.size
ratio = np.round(width/height,2)
print(width, height, ratio)

# width, height = 4032, 2268
print(width, height, ratio)

In [ ]:
img.show()

In [ ]:
img.size

In [ ]:
%%time

width, height = img.size

unique_colors = set()
for i in range(height):
    for j in range(width):
        pixel = img.getpixel((i, j))
        unique_colors.add(pixel)

print('Image info = ', img)
print('Unique color count = ', len(unique_colors))

In [ ]:
%%time

colors = {img.getpixel((i, j)) for i in range(height) for j in range(width)}
print(len(colors))

In [ ]:
class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, ctrl_case_dic, transform):
        '''
            ctrl_case_dic = {
                'ctrl': [root_train_one_pct, 'train_one_percent']
                'case': [root_train_inflamm, 'train_inflammation']
            }
        '''
        
        
        def get_images(name, dir):
            images = [x for x in os.listdir(dir)]
            print(f'Found {name} has {len(images)} images')
            return images
        
        self.images = {}
        self.class_names = ['ctrl', 'case']
        self.image_dirs = {}
        
        for s_class in self.class_names:
            dir, name = ctrl_case_dic[s_class]
            self.image_dirs[s_class] = dir
            self.images[s_class] = get_images(name, dir)
            
        self.ctrl_case_dic = ctrl_case_dic
        self.transform = transform
        
    
    def __len__(self):
        return sum([len(self.images[class_name]) for class_name in self.class_names])
    
    
    def __getitem__(self, index):
        class_name = random.choice(self.class_names)
        index = index % len(self.images[class_name])
        image_name = self.images[class_name][index]
        image_path = os.path.join(self.image_dirs[class_name], image_name)
        image = Image.open(image_path).convert('RGB')
        return self.transform(image), self.class_names.index(class_name)

# Image Transformations

In [ ]:
width, height = img.size
img_size = (height,width)
width, height 

In [ ]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=img_size),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=img_size),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Prepare DataLoader

In [ ]:
ctrl_case_dic = {'ctrl': [root_train_one_pct, 'train_one_percent'],
                 'case': [root_train_inflamm, 'train_inflammation'] }

train_dataset = ImageDataset(ctrl_case_dic, train_transform)

In [ ]:
test_dirs = {'ctrl': [root_test_one_pct, 'test_one_percent'],
             'case': [root_test_inflamm, 'test_inflammation'] }

test_dataset = ImageDataset(test_dirs, test_transform)

In [ ]:
batch_size = 3

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_test  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=True)

print('Number of training batches', len(dl_train))
print('Number of test batches', len(dl_test))

# Data Visualization

In [ ]:
class_names = train_dataset.class_names

def show_images(images, labels, preds):
    plt.figure(figsize=(128, 128))
    for i, image in enumerate(images):
        plt.subplot(1, 6, i + 1, xticks=[], yticks=[])
        image = image.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        image = image * std + mean
        image = np.clip(image, 0., 1.)
        plt.imshow(image)

        # print(i, preds[i], labels[i])
        col = 'red' if (preds[i] != labels[i]) else 'green'
            
        plt.xlabel(f'{class_names[int(labels[i].numpy())]}', fontsize=32)
        plt.ylabel(f'{class_names[int(preds[i].numpy())]}', color=col, fontsize=32)
    plt.tight_layout()
    plt.show()

In [ ]:
images, labels = next(iter(dl_train))
print(len(images), labels, labels)
show_images(images, labels, labels)

In [ ]:
images, labels = next(iter(dl_test))
print(len(images), labels, labels)
show_images(images, labels, labels)

In [ ]:
len(dl_test)

In [ ]:
for images, labels in dl_test:
    print(len(images), labels)

In [ ]:
width, height, width * height

# Creating the Model

In [ ]:
resnet18 = torchvision.models.resnet18(pretrained=True)
# print(resnet18)

In [ ]:
resnet18.fc = torch.nn.Linear(in_features=512, out_features=len(class_names))
loss_fn = torch.nn.CrossEntropyLoss()

# bad - optimizer = torch.optim.Adam(resnet18.parameters(), lr=1e-8)
optimizer = torch.optim.Adam(resnet18.parameters(), lr=1e-4)

from torch.optim.lr_scheduler import StepLR
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def show_preds():
    resnet18.eval()
    images, labels = next(iter(dl_test))
    outputs = resnet18(images)
    _, preds = torch.max(outputs, 1)
    print(len(images), labels, preds)
    show_images(images, labels, preds)

In [ ]:
show_preds()

# Training the Model

In [ ]:
root_data

In [ ]:
root_best = os.path.join(root_data, 'best_model')
create_dir(root_best)
fname_best_model= os.path.join(root_best, 'best_model_weights.pth')

fname_best_model

In [ ]:
def train(epochs):
    print('Starting training..')
    train_loss_list = []; mean_accuracy_list=[]; lr_list=[]
    count_resnet=-1; dic= {}; max_accuracy = 0; min_loss=9999

    for e in range(epochs):
        print('='*20)
        print(f'Starting epoch {e + 1}/{epochs}')
        print('='*20)

        train_loss = 0.
        val_loss = 0.

        resnet18.train() # set model to training phase
            
        accuracy_list = []

        for nTrain, (images, labels) in enumerate(dl_train):
            optimizer.zero_grad()
            outputs = resnet18(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            if nTrain % 10 == 0:
                print('Evaluating at step', nTrain)

                accuracy = 0
                resnet18.eval() # set model to eval phase
                for val_step, (images, labels) in enumerate(dl_test):
                    outputs = resnet18(images)
                    loss = loss_fn(outputs, labels)
                    val_loss += loss.item()

                    _, preds = torch.max(outputs, 1)
                    accuracy += np.sum((preds == labels).numpy())

                val_loss /= (val_step + 1)
                accuracy = accuracy/len(test_dataset)
                accuracy_list.append(accuracy)
                lr=optimizer.param_groups[0]["lr"]
                
                print(f'Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}, LR: {lr:.2e}')

                # show_preds()

                resnet18.train()

                if accuracy >= 0.80:
                    loss = train_loss / (nTrain + 1)
                    
                    if (accuracy >= max_accuracy) or (loss < min_loss):
                        
                        if (accuracy >= max_accuracy):
                            ''' PyTorch models store the learned parameters in an internal state dictionary, 
                                called state_dict. These can be persisted via the torch.save method '''
                            torch.save(resnet18.state_dict(), fname_best_model)

                        max_accuracy = accuracy
                        min_loss = loss
                        resnetx = resnet18.state_dict()
                        count_resnet += 1
                        dic[count_resnet]  = {}
                        dic[count_resnet]['lr'] = np.log10(lr)
                        dic[count_resnet]['accuracy'] = accuracy
                        dic[count_resnet]['loss'] = loss
                        dic[count_resnet]['resnet'] = resnetx
                        print(">>> %d) max accuracy %.2f, min loss %.5f "%(count_resnet, max_accuracy*100, min_loss) )

        mean_accuracy = np.mean(accuracy_list)
        train_loss /= (nTrain + 1)
        
        lr_list.append(lr)

        print(f'Training Loss: {train_loss:.4f}')
        train_loss_list.append(train_loss)
        mean_accuracy_list.append(mean_accuracy)
        
        scheduler.step()
        
    print('Training complete..')
    return train_loss_list, mean_accuracy_list, lr_list, dic

In [ ]:
%%time
train_loss_list, mean_accuracy_list, lr_list, dic = train(epochs=50)

# Final Results

In [ ]:
len(dic)

In [ ]:
import pandas as pd

df = pd.DataFrame(dic).T
print(len(df))
df = df.sort_values("accuracy", ascending=False)
df.head(3)

In [ ]:
fname = 'resnet_not_side.tsv'
df.to_csv(os.path.join(root_dir, fname), sep='\t')

In [ ]:
# current, last model
show_preds()

In [ ]:
# save current, last model
resnet18_backup = copy.deepcopy(resnet18)

# load best model
#resnet18.load_state_dict(torch.load(fname_best_model))
#resnet18.eval()

In [ ]:
resnet18.load_state_dict(df.iloc[0].resnet)

images, labels = next(iter(dl_test))
outputs = resnet18(images)
_, preds = torch.max(outputs, 1)
print(len(images), labels, preds)
show_images(images, labels, preds)

In [ ]:
resnet18.load_state_dict(df.iloc[1].resnet)

for images, labels in dl_test:
    outputs = resnet18(images)
    _, preds = torch.max(outputs, 1)
    print(len(images), labels, preds)
    show_images(images, labels, preds)

In [ ]:
df.iloc[0].accuracy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline


plt.plot(train_loss_list, linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("loss", fontsize=14)
plt.show()

In [ ]:
plt.plot([np.log10(x) for x in lr_list], linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("lr", fontsize=14)
plt.show()

In [ ]:
plt.plot(mean_accuracy_list, linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("accuracy in %", fontsize=14)
plt.show()